Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Install Dependencies

In [ ]:
!pip -q install --upgrade kaggle torch torchvision torchaudio scikit-learn matplotlib tqdm kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 128.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12

Set up kagglehub credentials

In [ ]:
import kagglehub
kagglehub.login()

Install Dataset

In [ ]:
import kagglehub
path = kagglehub.dataset_download("mshrestha/plant-disease-augmented-dataset")

100%|██████████| 8.78G/8.78G [01:43<00:00, 91.3MB/s]

Extracting files...


Paths and Project Setup

In [ ]:
from pathlib import Path

DATA_SRC = Path(path)

PROJECT     = "plant_efficientnetb0"
DRIVE_ROOT  = Path("/content/drive/MyDrive")
CKPT_DIR    = DRIVE_ROOT / PROJECT / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

WORK_ROOT = Path("/content/plant_split")
TRAIN = WORK_ROOT / "train"
VAL   = WORK_ROOT / "val"

print("Data source:", DATA_SRC)
print("Checkpoints:", CKPT_DIR)
print("Split root :", WORK_ROOT)


Data source: /root/.cache/kagglehub/datasets/mshrestha/plant-disease-augmented-dataset/versions/1
Checkpoints: /content/drive/MyDrive/plant_efficientnetb0/checkpoints
Split root : /content/plant_split


In [ ]:
import os, random

In [ ]:
def is_image(p: Path) -> bool:
  return p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".tif",".tiff",".webp"}

Split the data into training and validation

In [ ]:
def make_split_symlinks(src: Path, train: Path, val: Path, seed=42, train_ratio=0.8):
  random.seed(seed)

  # Make the folders if they don't exist already
  train.mkdir(parents=True, exist_ok=True)
  val.mkdir(parents=True, exist_ok=True)

  # Class of the plant diseases
  class_dirs = [d for d in src.iterdir() if d.is_dir()]
  assert len(class_dirs) >= 2, "Expected multiple class folders under DATA_SRC."

  # Going through each photos in plant disease folders
  for cls_dir in sorted(class_dirs):
    imgs = [p for p in cls_dir.iterdir() if p.is_file() and is_image(p)]
    random.shuffle(imgs)
    k = int(len(imgs)*train_ratio)

    # Make the train, val directories if they don't exist already
    (train/cls_dir.name).mkdir(parents=True, exist_ok=True)
    (val/cls_dir.name).mkdir(parents=True, exist_ok=True)

    for i, p in enumerate(imgs):
      dst_root = train if i < k else val
      link_path = dst_root/cls_dir.name/p.name
      if not link_path.exists():
        try:
          os.symlink(p, link_path)
        except FileExistsError:
          pass


In [ ]:
if not (TRAIN.exists() and any(TRAIN.iterdir())):
  make_split_symlinks(DATA_SRC, TRAIN, VAL)
  print("Symlink split created.")
else:
  print("Split already exists; reusing")

print("Train classes:")
for d in TRAIN.iterdir():
  if d.is_dir():
    print("\t",d.name)
print("Val classes:")
for d in VAL.iterdir():
  if d.is_dir():
    print("\t",d.name)

Symlink split created.
Train classes:
	 POTATO_EARLY_BLIGHT
	 STRAWBERRY_HEALTHY
	 APPLE_HEALTHY
	 TOMATO_BACTERIAL_SPOT
	 RICE_HEALTHY
	 TOMATO_HEALTHY
	 BANANA_HEALTHY
	 APPLE_SCAB
	 STRAWBERRY_LEAF_SCORCH
	 TOMATO_LEAF_MOLD
	 TEA_ALGAL_SPOT
	 BANANA_SIGATOKA
	 CORN_LEAF_GRAY_SPOT
	 TOMATO_LATE_BLIGHT
	 RICE_LEAF_BLIGHT
	 PEPPER_BELL_HEALTHY
	 TOMATO_TARGET_SPOT
	 POTATO_LATE_BLIGHT
	 TOMATO_MOSAIC_VIRUS
	 BANANA_PANAMA
	 CORN_LEAF_RUST
	 PEPPER_BELL_BACTERIAL_SPOT
	 TEA_RED_LEAF_SPOT
	 POTATO_HEALTHY
	 CORN_HEALTHY
	 TOMATO_EARLY_BLIGHT
	 TEA_HEALTHY
	 TOMATO_SEPTORIA_LEAF_SPOT
	 RICE_LEAF_BLAST
	 APPLE_ROT
	 APPLE_RUST
	 RICE_LEAF_BROWN_SPOT
	 TEA_BROWN_BLIGHT
	 CORN_LEAF_BLIGHT
Val classes:
	 POTATO_EARLY_BLIGHT
	 STRAWBERRY_HEALTHY
	 APPLE_HEALTHY
	 TOMATO_BACTERIAL_SPOT
	 RICE_HEALTHY
	 TOMATO_HEALTHY
	 BANANA_HEALTHY
	 APPLE_SCAB
	 STRAWBERRY_LEAF_SCORCH
	 TOMATO_LEAF_MOLD
	 TEA_ALGAL_SPOT
	 BANANA_SIGATOKA
	 CORN_LEAF_GRAY_SPOT
	 TOMATO_LATE_BLIGHT
	 RICE_LEAF_BLIGHT
	 PEPPER_

DataLoaders (Augmentations + Normalization)

In [ ]:
import json, numpy as np, torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms

In [ ]:
IMG = 224
BATCH = 32
NUM_WORKERS = 2

train_tfms = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.15, 0.05),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
val_tfms = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

train_ds = datasets.ImageFolder(str(TRAIN), transform=train_tfms)
val_ds = datasets.ImageFolder(str(VAL), transform=val_tfms)
class_names = train_ds.classes

with open(CKPT_DIR / "classes.json", "w") as f:
  json.dump(class_names, f)

# Handle impbalance with a WeightedRandomSampler
counts = np.bincount([y for _, y in train_ds.samples])
weights = 1.0 /np.maximum(counts, 1)
sample_weights = [weights[y] for _, y in train_ds.samples]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_dl = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

len(train_ds), len(val_ds), len(class_names)

(119894, 29994, 34)

Model, optimizer, scheduler

In [ ]:
from torchvision import models
from torch import nn

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

'cuda'

In [ ]:
weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
model = models.efficientnet_b0(weights=weights)
# model.fc = nn.Linear(model.fc.in_features, len(class_names))
# model.to(DEVICE)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, len(class_names))

model.to(DEVICE)

LR = 3e-4 # Learning Rate
WD = 1e-4 # Weight Decay
EPOCHS = 20

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda"))

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 157MB/s]
/tmp/ipython-input-1643905411.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda"))


Checkpoint helpers (Drive-persistent) + auto-resume

In [ ]:
import torch
from pathlib import Path

In [ ]:
BEST_PATH = CKPT_DIR / "best.pt"
LAST_PATH = CKPT_DIR / "last.pt"

In [ ]:
def save_ckpt(epoch: int, best_acc: float, path: Path):
  torch.save({
      "epoch": epoch,
      "best_acc": best_acc,
      "model_state": model.state_dict(),
      "optimizer_state": optimizer.state_dict(),
      "scheduler_state": scheduler.state_dict(),
      "classes": class_names,
      "img_size": IMG
  }, path)

In [ ]:
def load_ckpt(path: Path):
  try:
    ckpt = torch.load(path, map_location='cpu', weights_only=True)
  except TypeError:
    ckpt = torch.load(path, map_location = 'cpu')
  model.load_state_dict(ckpt["model_state"])
  optimizer.load_state_dict(ckpt["optimizer_state"])
  scheduler.load_state_dict(ckpt["scheduler_state"])
  return ckpt.get("epoch", 0), ckpt.get("best_acc", 0.0)

start_epoch, best_acc = 0, 0.0
if LAST_PATH.exists():
  start_epoch, best_acc = load_ckpt(LAST_PATH)
  print(f"Resumed from {LAST_PATH.name}: epoch={start_epoch}, best_acc={best_acc:.4f}")
else:
  print("Starting fresh (no last checkpoint).")

Starting fresh (no last checkpoint).


Train + Validate (saves last.pt per epoch, best.pt on improvement)

In [ ]:
from tqdm import tqdm

In [ ]:
for epoch in range(start_epoch, EPOCHS):
  # Train
  model.train()
  tr_correct = tr_total = 0
  for x, y in tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS} [train]"):
    x, y = x.to(DEVICE), y.to(DEVICE)
    optimizer.zero_grad(set_to_none=True)
    with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
      logits = model(x)
      loss = criterion(logits, y)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    tr_correct += (logits.argmax(1) == y).sum().item()
    tr_total += x.size(0)

  train_acc = tr_correct / tr_total
  scheduler.step()

  # Validate
  model.eval()
  v_correct = v_total = 0
  with torch.no_grad():
    for x, y in tqdm(val_dl, desc=f"Epoch {epoch + 1}/{EPOCHS} [val]"):
      x, y = x.to(DEVICE), y.to(DEVICE)
      with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
        logits = model(x)
        loss = criterion(logits, y)
      v_correct += (logits.argmax(1) == y).sum().item()
      v_total += x.size(0)
  val_acc = v_correct/v_total

  # Checkpoints
  save_ckpt(epoch+1, best_acc, LAST_PATH)
  if val_acc > best_acc:
    best_acc = val_acc
    save_ckpt(epoch+1, best_acc, BEST_PATH)
    print(f"New Best {best_acc:.4f} saved at {BEST_PATH}")

Epoch 1/20 [train]:   0%|          | 0/3747 [00:00<?, ?it/s]/tmp/ipython-input-2251293505.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
Epoch 1/20 [val]:   0%|          | 0/938 [00:00<?, ?it/s]/tmp/ipython-input-2251293505.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
Epoch 1/20 [val]: 100%|██████████| 938/938 [02:24<00:00,  6.47it/s]


New Best 0.9846 saved at /content/drive/MyDrive/plant_efficientnetb0/checkpoints/best.pt


Epoch 2/20 [val]: 100%|██████████| 938/938 [02:21<00:00,  6.64it/s]


New Best 0.9901 saved at /content/drive/MyDrive/plant_efficientnetb0/checkpoints/best.pt


Epoch 6/20 [val]: 100%|██████████| 938/938 [02:27<00:00,  6.34it/s]


...

Epoch 9/20 [val]: 100%|██████████| 938/938 [02:22<00:00,  6.58it/s]


New Best 0.9945 saved at /content/drive/MyDrive/plant_efficientnetb0/checkpoints/best.pt


Epoch 10/20 [val]: 100%|██████████| 938/938 [02:22<00:00,  6.59it/s]


New Best 0.9958 saved at /content/drive/MyDrive/plant_efficientnetb0/checkpoints/best.pt


Epoch 12/20 [train]:  86%|████████▌ | 3229/3747 [18:15<04:39,  1.85it/s]

Final report + confusion matrix

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader

In [ ]:
# Load BEST for reporting
if BEST_PATH.exists():
    load_ckpt(BEST_PATH)

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x,y in DataLoader(val_ds, batch_size=64, shuffle=False):
        x = x.to(DEVICE)
        logits = model(x)
        y_true.extend(y.tolist())
        y_pred.extend(logits.argmax(1).cpu().tolist())

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
cm = confusion_matrix(y_true, y_pred)
np.savetxt(CKPT_DIR / "confusion_matrix.csv", cm, fmt="%d", delimiter=",")
print("Saved:", CKPT_DIR / "confusion_matrix.csv")


Inference

In [ ]:
from PIL import Image
from torchvision import transforms

In [ ]:
if BEST_PATH.exists():
    load_ckpt(BEST_PATH)

prep = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
])

def predict_image(path:str):
    im = Image.open(path).convert("RGB")
    x = prep(im).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(x)
        prob = torch.softmax(logits,1).max().item()
        idx = logits.argmax(1).item()
    return class_names[idx], prob

# label, conf = predict_image("/content/leaf.jpg")
# print(label, conf)